<a href="https://colab.research.google.com/github/ngale68/bda-final-individual-project/blob/main/part3_spark_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Team 3: Analytics with Apache Spark

### Task 1: Start a Spark session, load the data, and print schema, row count, and column names.

In [18]:
!pip install pyspark
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, expr, lit

spark = SparkSession.builder \
    .appName("MarketDataAnalysis") \
    .getOrCreate()

print("Spark session started")

csv_file_path = "/content/data/clean/cleaned_market_data.csv"

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_file_path)

print(f"Loaded file: {csv_file_path}")

# Cache the DataFrame to prevent re-reading the CSV file for subsequent operations
df.cache()

row_count = df.count()
print(f"Row count: {row_count}")
print("Columns: " + ", ".join(df.columns))
print("Schema:")
df.printSchema()

for c in ["open_time", "close", "volume"]:
    if c in df.columns:
        print(f"{c}: {df.schema[c].dataType.simpleString()}")

Spark session started
Loaded file: /content/data/clean/cleaned_market_data.csv
Row count: 9628
Columns: symbol, interval, open_time, open, high, low, close, volume, close_time, quote_volume, trade_count, open_time__orig, close_time__orig, price_range, price_change, percent_change, candle_direction
Schema:
root
 |-- symbol: string (nullable = true)
 |-- interval: string (nullable = true)
 |-- open_time: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- close_time: timestamp (nullable = true)
 |-- quote_volume: double (nullable = true)
 |-- trade_count: double (nullable = true)
 |-- open_time__orig: string (nullable = true)
 |-- close_time__orig: string (nullable = true)
 |-- price_range: double (nullable = true)
 |-- price_change: double (nullable = true)
 |-- percent_change: double (nullable = true)
 |-- candle_direction: s

### Task 2: Register the DataFrame as a temporary SQL view named `market_data` and run a small test query.

In [19]:
df.createOrReplaceTempView("market_data")
print("Temporary SQL view created: market_data")

test_query_result = spark.sql("SELECT * FROM market_data LIMIT 10")
num_rows_returned = test_query_result.count()
print(f"Test query returned rows: {num_rows_returned}")

Temporary SQL view created: market_data
Test query returned rows: 10


### Task 3: Add or verify `price_range`, `price_change`, `percent_change`, and `candle_direction` in Spark.

In [20]:
required_columns = ["price_range", "price_change", "percent_change", "candle_direction"]

# Check if the columns exist in the DataFrame
existing_columns = [c for c in required_columns if c in df.columns]
missing_columns = [c for c in required_columns if c not in df.columns]

if not missing_columns:
    print(f"Created/verified columns:\n{', '.join(existing_columns)}")
else:
    print(f"The following required columns are missing: {', '.join(missing_columns)}")

# Show a small preview to prove the columns exist and have correct values
print("\nExample row:")
for row in df.select("symbol", "price_range", "price_change", "percent_change", "candle_direction") \
  .limit(1) \
  .collect():
    print(f"symbol={row.symbol} price_range={row.price_range:.2f} price_change={row.price_change:.2f} percent_change={row.percent_change:.2f} candle_direction={row.candle_direction}")

Created/verified columns:
price_range, price_change, percent_change, candle_direction

Example row:
symbol=DOGEUSDT price_range=0.00 price_change=0.00 percent_change=0.33 candle_direction=up


### Task 4: Create time features and run full-dataset Spark SQL queries for average close price, average volume, and row count by symbol. Compare with Team 2's sample.

In [21]:
from pyspark.sql.functions import to_date, hour, date_format

print("Creating time features:")

df = df.withColumn("trade_date", to_date(col("open_time")))
df = df.withColumn("trade_hour", hour(col("open_time")))
df = df.withColumn("day_of_week", date_format(col("open_time"), "EEE"))

df.createOrReplaceTempView("market_data")

print("Created time features:")
print("trade_date, trade_hour, day_of_week")

print("\nExample row with new time features:")
for row in df.select("open_time", "trade_date", "trade_hour", "day_of_week") \
  .limit(1) \
  .collect():
    print(f"open_time={row.open_time} trade_date={row.trade_date} trade_hour={row.trade_hour} day_of_week={row.day_of_week}")

print("\nAverage close by symbol:")
avg_close_by_symbol = spark.sql(
    "SELECT symbol, AVG(close) as avg_close_price FROM market_data GROUP BY symbol ORDER BY symbol"
)
avg_close_by_symbol.show(truncate=False)

print("\nAverage volume by symbol:")
avg_volume_by_symbol = spark.sql(
    "SELECT symbol, AVG(volume) as avg_volume FROM market_data GROUP BY symbol ORDER BY symbol"
)
avg_volume_by_symbol.show(truncate=False)

print("\nRow count by symbol:")
row_count_by_symbol = spark.sql(
    "SELECT symbol, COUNT(*) as total_rows FROM market_data GROUP BY symbol ORDER BY symbol"
)
row_count_by_symbol.show(truncate=False)

print("\nFull Spark result uses all cleaned rows, not only 50 sample rows. Therefore, it is more reliable and representative of the entire dataset's trends and characteristics, reducing the risk of bias from a small sample.")

Creating time features:
Created time features:
trade_date, trade_hour, day_of_week

Example row with new time features:
open_time=2026-06-26 22:00:00 trade_date=2026-06-26 trade_hour=22 day_of_week=Fri

Average close by symbol:
+--------+-------------------+
|symbol  |avg_close_price    |
+--------+-------------------+
|ADAUSDT |0.18051058091286326|
|AVAXUSDT|7.178663865546215  |
|BNBUSDT |603.6111610878667  |
|BTCUSDT |65163.370317796594 |
|DOGEUSDT|0.08596786235662145|
|DOTUSDT |0.9952052631578948 |
|ETHUSDT |1759.1359200841255 |
|LINKUSDT|8.07234182590234   |
|SOLUSDT |73.57619791666657  |
|XRPUSDT |1.1673705141657924 |
+--------+-------------------+


Average volume by symbol:
+--------+--------------------+
|symbol  |avg_volume          |
+--------+--------------------+
|ADAUSDT |7385749.485952131   |
|AVAXUSDT|115074.52493684227  |
|BNBUSDT |7475.63566945606    |
|BTCUSDT |794.9618609168452   |
|DOGEUSDT|2.6105854941238195E7|
|DOTUSDT |279503.51987328386  |
|ETHUSDT |13573.417634

### Task 5: Create a volatility ranking by symbol.

In [22]:
from pyspark.sql.functions import avg, stddev, min, max, row_number
from pyspark.sql.window import Window

print("Creating volatility ranking:")

volatility_df = df.groupBy("symbol").agg(
    avg("price_range").alias("avg_price_range"),
    stddev("price_range").alias("stddev_price_range"),
    min("price_range").alias("min_price_range"),
    max("price_range").alias("max_price_range")
)

volatility_df.createOrReplaceTempView("volatility_data")

window_spec = Window.orderBy(col("avg_price_range").desc(), col("stddev_price_range").desc())
ranked_volatility_df = volatility_df.withColumn("rank", row_number().over(window_spec))

print("Volatility ranking:")
ranked_volatility_df.select("rank", "symbol", "avg_price_range", "stddev_price_range") \
    .orderBy("rank") \
    .show(truncate=False)

Creating volatility ranking:
Volatility ranking:
+----+--------+--------------------+---------------------+
|rank|symbol  |avg_price_range     |stddev_price_range   |
+----+--------+--------------------+---------------------+
|1   |BTCUSDT |440.9909860664517   |311.84586597455734   |
|2   |ETHUSDT |15.702728249194411  |11.42541711293583    |
|3   |BNBUSDT |4.539229144667375   |3.427531199561864    |
|4   |SOLUSDT |0.797333333333333   |0.5039596180351847   |
|5   |AVAXUSDT|0.08559042553191494 |0.055752164519362164 |
|6   |LINKUSDT|0.0820096566523606  |0.05168772334372744  |
|7   |DOTUSDT |0.012052406417112296|0.007624233881234349 |
|8   |XRPUSDT |0.010729915433403802|0.007373896717277353 |
|9   |ADAUSDT |0.002265601703940369|0.0014072303905572712|
|10  |DOGEUSDT|8.250579557428882E-4|5.656944891692045E-4 |
+----+--------+--------------------+---------------------+



### Task 6: Create an activity ranking by symbol.

In [23]:
from pyspark.sql.functions import sum, avg, col, row_number
from pyspark.sql.window import Window

print("Creating activity ranking:")

activity_df = df.groupBy("symbol").agg(
    sum("trade_count").alias("total_trades"),
    sum("quote_volume").alias("total_quote_volume"),
    avg("volume").alias("average_volume")
)

window_spec_activity = Window.orderBy(col("total_trades").desc(), col("total_quote_volume").desc())
ranked_activity_df = activity_df.withColumn("rank", row_number().over(window_spec_activity))

print("Activity ranking:")
ranked_activity_df.select("rank", "symbol", "total_trades", "total_quote_volume", "average_volume") \
    .orderBy("rank") \
    .show(truncate=False)

Creating activity ranking:
Activity ranking:
+----+--------+------------+---------------------+--------------------+
|rank|symbol  |total_trades|total_quote_volume   |average_volume      |
+----+--------+------------+---------------------+--------------------+
|1   |BTCUSDT |1.49663456E8|4.621654539651874E10 |794.9618609168452   |
|2   |ETHUSDT |1.4343186E8 |2.1837559870542114E10|13573.417634952497  |
|3   |SOLUSDT |3.8475756E7 |7.496970256561802E9  |112624.92109832642  |
|4   |BNBUSDT |3.5935949E7 |4.3022580159782095E9 |7475.63566945606    |
|5   |XRPUSDT |2.8975256E7 |4.3881985724485235E9 |4104006.104070981   |
|6   |DOGEUSDT|2.4616093E7 |2.1205625589282777E9 |2.6105854941238195E7|
|7   |AVAXUSDT|9679412.0   |7.402225869373511E8  |115074.52493684227  |
|8   |LINKUSDT|5811762.0   |6.916789360176103E8  |97173.2464936441    |
|9   |ADAUSDT |5164966.0   |1.2195553271509213E9 |7385749.485952131   |
|10  |DOTUSDT |1468833.0   |2.5773244191342008E8 |279503.51987328386  |
+----+--------+----

### Task 7: Analyze activity by time to find the busiest hour and busiest date.

In [24]:
print("Analyzing activity by time:")

busiest_hour_trades = spark.sql(
    "SELECT trade_hour, SUM(trade_count) as total_trades FROM market_data GROUP BY trade_hour ORDER BY total_trades DESC LIMIT 1"
)
print("\nBusiest hour by trades:")
busiest_hour_trades.show()

busiest_date_quote_volume = spark.sql(
    "SELECT trade_date, SUM(quote_volume) as total_quote_volume FROM market_data GROUP BY trade_date ORDER BY total_quote_volume DESC LIMIT 1"
)
print("\nBusiest date by quote volume:")
busiest_date_quote_volume.show()

print("\nExplanation: The market activity was highest around the identified busiest hour and date, as these periods recorded the highest number of trades and total quote volume, respectively.")

Analyzing activity by time:

Busiest hour by trades:
+----------+------------+
|trade_hour|total_trades|
+----------+------------+
|        14|  3.427322E7|
+----------+------------+


Busiest date by quote volume:
+----------+-------------------+
|trade_date| total_quote_volume|
+----------+-------------------+
|2026-06-05|6.102736676427874E9|
+----------+-------------------+


Explanation: The market activity was highest around the identified busiest hour and date, as these periods recorded the highest number of trades and total quote volume, respectively.


### Task 8: Build a final ranked market summary table and save it.

In [25]:
import os
from pyspark.sql.functions import avg, sum, count, col, lit
import shutil
import glob

results_dir = "results"
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

final_csv_file_path = os.path.join(results_dir, "spark_market_summary.csv")
temp_output_dir = os.path.join(results_dir, "temp_spark_output")

candle_counts_df = df.groupBy("symbol", "candle_direction").agg(count("*").alias("count"))

pivoted_candle_counts_df = candle_counts_df.groupBy("symbol") \
    .pivot("candle_direction", ["up", "down", "flat"]) \
    .agg(sum("count")) \
    .fillna(0) \
    .withColumnRenamed("up", "up_candles") \
    .withColumnRenamed("down", "down_candles") \
    .withColumnRenamed("flat", "flat_candles")

avg_percent_change_df = df.groupBy("symbol").agg(avg("percent_change").alias("avg_percent_change"))

summary_df = ranked_volatility_df.alias("rvd") \
    .join(ranked_activity_df.alias("rad"), col("rvd.symbol") == col("rad.symbol"), "inner") \
    .select(
        col("rvd.symbol"),
        col("rvd.rank").alias("volatility_rank"),
        col("rvd.avg_price_range"),
        col("rad.rank").alias("activity_rank"),
        col("rad.total_trades"),
        col("rad.total_quote_volume"),
        col("rad.average_volume") # This is average volume from activity_df
    )

summary_df = summary_df.join(row_count_by_symbol.alias("rcbs"), summary_df["symbol"] == col("rcbs.symbol"), "inner") \
    .select(
        summary_df["*"] ,
        col("rcbs.total_rows").alias("total_records")
    )

summary_df = summary_df.join(avg_percent_change_df.alias("apcd"), summary_df["symbol"] == col("apcd.symbol"), "inner") \
    .select(
        summary_df["*"] ,
        col("apcd.avg_percent_change")
    )

summary_df = summary_df.join(pivoted_candle_counts_df.alias("pccd"), summary_df["symbol"] == col("pccd.symbol"), "inner") \
    .select(
        summary_df["*"] ,
        col("pccd.up_candles"),
        col("pccd.down_candles"),
        col("pccd.flat_candles")
    )

final_summary_df = summary_df.select(
    col("symbol"),
    col("total_records"),
    col("average_volume"),
    col("total_trades"),
    col("avg_percent_change"),
    col("avg_price_range"),
    col("up_candles"),
    col("down_candles"),
    col("flat_candles"),
    col("volatility_rank"),
    col("activity_rank")
).orderBy(col("activity_rank"))

print("\nFinal ranked market summary created:")
final_summary_df.show(truncate=False)
print(f"Rows in summary: {final_summary_df.count()}")

final_summary_df.coalesce(1).write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv(temp_output_dir)

csv_files = glob.glob(f"{temp_output_dir}/part-*.csv")
if csv_files:
    actual_csv_file_in_temp_dir = csv_files[0]
    os.rename(actual_csv_file_in_temp_dir, final_csv_file_path)
    shutil.rmtree(temp_output_dir, ignore_errors=True)
    print(f"Saved: {final_csv_file_path}")
else:
    print("Error: Could not find the generated CSV file.")


# Interpretation for Zehra
top_activity_symbol = ranked_activity_df.orderBy(col("rank")).limit(1).collect()[0].symbol
top_volatility_symbol = ranked_volatility_df.orderBy(col("rank")).limit(1).collect()[0].symbol

print(f"\nTop activity symbol: {top_activity_symbol}")
print(f"Top volatility symbol: {top_volatility_symbol}")

interpretation_message = f"Interpretation for Zehra:\n{top_activity_symbol} was the most active symbol (ranked #1), indicating strong market interest. Meanwhile, {top_volatility_symbol} was the most volatile (also ranked #1), showing the widest average price range and suggesting significant price swings. These points could be key for detailed reporting."
print(interpretation_message)

spark.stop()
print("\nSpark session stopped.")


Final ranked market summary created:
+--------+-------------+--------------------+------------+---------------------+--------------------+----------+------------+------------+---------------+-------------+
|symbol  |total_records|average_volume      |total_trades|avg_percent_change   |avg_price_range     |up_candles|down_candles|flat_candles|volatility_rank|activity_rank|
+--------+-------------+--------------------+------------+---------------------+--------------------+----------+------------+------------+---------------+-------------+
|BTCUSDT |952          |794.9618609168452   |1.49663456E8|-0.012180285673263139|440.9909860664517   |450       |484         |1           |1              |1            |
|ETHUSDT |961          |13573.417634952497  |1.4343186E8 |-0.02365321035522852 |15.702728249194411  |460       |484         |1           |2              |2            |
|SOLUSDT |966          |112624.92109832642  |3.8475756E7 |0.004994923402026134 |0.797333333333333   |474       |471  